# Task 3: Partition Pruning Experiment (Queries 2, 3, 4, 5)

- February 2024 queries are run on a date-partitioned Gold table and an identical unpartitioned Delta copy.

## Connect to Spark

In [5]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import (
    DEFAULT_INTEGRATED_TABLE_PATH,
    INTEGRATED_TABLE_VIEW,
    execute_registered_query,
    register_integrated_taxi_trips,
)

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-PartitionPruning-Query3')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

## Define the two layouts

- February is used because the data covers January-March 2024. 
- Partition prunnig by skipping January and March.

In [6]:
PARTITIONED_TABLE_PATH = DEFAULT_INTEGRATED_TABLE_PATH
UNPARTITIONED_TABLE_PATH = (
    's3a://delta-data/gold/gold_integrated_taxi_trips_unpartitioned'
)
QUERY_NAMES = (
    'query_02_2024',
    'query_04_2024',
    'query_03_air_quality_vs_demand_february_2024',
    'query_05_peak_travel_hours_february_2024',
)
REPETITIONS = 9

## Create the unpartitioned layout

In [7]:
partitioned_table = spark.read.format('delta').load(PARTITIONED_TABLE_PATH)

(
    partitioned_table.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(UNPARTITIONED_TABLE_PATH)
)

unpartitioned_table = (
    spark.read.format('delta').load(UNPARTITIONED_TABLE_PATH)
)


assert partitioned_table.count() == unpartitioned_table.count()
print('Unpartitioned layout created with the same number of rows.')

Unpartitioned layout created with the same number of rows.


## Measure both layouts

Each layout receives one warm-up followed by three timed runs.

In [8]:
spark.read.format("delta") \
    .load("s3a://delta-data/silver/silver_weather_conditions") \
    .createOrReplaceTempView("weather_conditions")

In [10]:
def measure_layout(table_path):
    spark.catalog.clearCache()
    register_integrated_taxi_trips(spark, table_path)
    return {
        query_name: measure_registered_query(spark, query_name, 9)
        for query_name in QUERY_NAMES
    }

partitioned_results = measure_layout(PARTITIONED_TABLE_PATH)
unpartitioned_results = measure_layout(UNPARTITIONED_TABLE_PATH)

for query_name in QUERY_NAMES:
    assert rows_match(
        partitioned_results[query_name]['rows'],
        unpartitioned_results[query_name]['rows'],
    ), f'Partitioned and unpartitioned results differ for {query_name}.'

print('Both layouts return the same results within float tolerance.')

Both layouts return the same results within float tolerance.


## Partitioned physical plans

- `PartitionFilters` contains the February 2024 range.

In [11]:
register_integrated_taxi_trips(spark, PARTITIONED_TABLE_PATH)
for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_02_2024 ---
== Physical Plan ==
AdaptiveSparkPlan (14)
+- Sort (13)
   +- Exchange (12)
      +- HashAggregate (11)
         +- Exchange (10)
            +- HashAggregate (9)
               +- Project (8)
                  +- BroadcastHashJoin Inner BuildRight (7)
                     :- Project (3)
                     :  +- Filter (2)
                     :     +- Scan parquet  (1)
                     +- BroadcastExchange (6)
                        +- Filter (5)
                           +- Scan parquet  (4)


(1) Scan parquet 
Output [3]: [trip_distance#40057, coco_pu#40097, pickup_month#40072]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_month#40072), (pickup_month#40072 = 2024-02)]
PushedFilters: [IsNotNull(coco_pu)]
ReadSchema: struct<trip_distance:double,coco_pu:int>

(2) Filter
Input [3]: [trip_distance#40057, coco_pu#40097, pickup_month#40072]
Condition : isnotnull(coco_pu#40

## Unpartitioned physical plans

- It does not contain a `PartitionFilters` entry.

In [12]:
register_integrated_taxi_trips(spark, UNPARTITIONED_TABLE_PATH)
for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_02_2024 ---
== Physical Plan ==
AdaptiveSparkPlan (14)
+- Sort (13)
   +- Exchange (12)
      +- HashAggregate (11)
         +- Exchange (10)
            +- HashAggregate (9)
               +- Project (8)
                  +- BroadcastHashJoin Inner BuildRight (7)
                     :- Project (3)
                     :  +- Filter (2)
                     :     +- Scan parquet  (1)
                     +- BroadcastExchange (6)
                        +- Filter (5)
                           +- Scan parquet  (4)


(1) Scan parquet 
Output [3]: [trip_distance#40852, pickup_month#40867, coco_pu#40892]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips_unpartitioned]
PushedFilters: [IsNotNull(pickup_month), EqualTo(pickup_month,2024-02), IsNotNull(coco_pu)]
ReadSchema: struct<trip_distance:double,pickup_month:string,coco_pu:int>

(2) Filter
Input [3]: [trip_distance#40852, pickup_month#40867, coco_pu#40892]
Condition : ((isnotnull(

## Benchmark summary

In [13]:
summary = []
for query_name in QUERY_NAMES:
    partitioned = partitioned_results[query_name]
    unpartitioned = unpartitioned_results[query_name]

    summary.append({
        'query': query_name,
        'partitioned_samples_seconds': [
            round(value, 3) for value in partitioned['samples_seconds']
        ],

        'partitioned_median_seconds': round(partitioned['median_seconds'], 3),
        'unpartitioned_samples_seconds': [
            round(value, 3) for value in unpartitioned['samples_seconds']
        ],
        'unpartitioned_median_seconds': round(unpartitioned['median_seconds'], 3),
        'partitioned_speedup_factor': round(
            unpartitioned['median_seconds'] / partitioned['median_seconds'],
            2,
        ),
    })
summary

[{'query': 'query_02_2024',
  'partitioned_samples_seconds': [0.816,
   0.83,
   0.816,
   0.911,
   0.762,
   0.771,
   0.818,
   0.915,
   0.906],
  'partitioned_median_seconds': 0.818,
  'unpartitioned_samples_seconds': [0.86,
   0.829,
   0.863,
   0.78,
   0.804,
   0.819,
   0.859,
   0.783,
   0.844],
  'unpartitioned_median_seconds': 0.829,
  'partitioned_speedup_factor': 1.01},
 {'query': 'query_04_2024',
  'partitioned_samples_seconds': [0.593,
   0.702,
   0.605,
   0.637,
   0.595,
   0.54,
   0.58,
   0.548,
   0.604],
  'partitioned_median_seconds': 0.595,
  'unpartitioned_samples_seconds': [0.658,
   0.679,
   0.591,
   0.655,
   0.71,
   0.693,
   0.72,
   0.614,
   0.751],
  'unpartitioned_median_seconds': 0.679,
  'partitioned_speedup_factor': 1.14},
 {'query': 'query_03_air_quality_vs_demand_february_2024',
  'partitioned_samples_seconds': [0.502,
   0.606,
   0.589,
   0.514,
   0.584,
   0.666,
   0.588,
   0.531,
   0.538],
  'partitioned_median_seconds': 0.584,
 

## Cleanup

In [14]:
spark.catalog.clearCache()
spark.stop()